In [1]:
# 7-4-2026

In [4]:
import glob
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from scipy.stats import wasserstein_distance
from itertools import combinations
from scipy.stats import kendalltau

In [5]:
train_x_dir = "../transfer-matrix/train_X"

# glob for domain csvs
domain_files = glob.glob(os.path.join(train_x_dir, "domain_*.csv"))

In [7]:
base_features = [
    "fwi_mean", "lccs_class_1", "lccs_class_2", "lccs_class_3", "lccs_class_4",
    "lccs_class_6", "lccs_class_7", "ndvi", "pop_dens", "rel_hum", "skt",
    "ssr", "ssrd", "swvl1", "swvl2", "swvl3", "swvl4", "t2m_max", "t2m_mean",
    "t2m_min", "tp", "vpd", "ws10"
]

# fixed seed
SEED = 5
N_SAMPLES = 1000

In [ ]:
domain_samples = {}

for f in domain_files:
    # pull domain id out of filename, ex "domain_7.csv": 7
    domain_id = int(os.path.basename(f).replace("domain_", "").replace(".csv", ""))

    df = pd.read_csv(f)

    # keep only the base features, drops target col and anything else automatically
    df = df[base_features]

    # sample 1000 rows, use min for fallback (not gonna happen though)
    n = min(N_SAMPLES, len(df))
    sample = df.sample(n=n, random_state=SEED)

    domain_samples[domain_id] = sample

In [ ]:
len(domain_samples) # good

34

In [10]:
# stack all domans together to fit one scaler, not perdomain
#   per domain scaling would hide distribution differences between domains which is
#   exactly what wasserstein is suposed to detect, one scaler keeps everything on the same ruler
all_rows = pd.concat(domain_samples.values(), axis=0)

scaler = StandardScaler().fit(all_rows)

# apply same scaler to each domain, store back as numpy arrays for wasserstein step
domain_scaled = {
    domain_id: scaler.transform(sample)
    for domain_id, sample in domain_samples.items()
}

In [ ]:
len(all_rows) # 34x1000, good

34000

In [12]:
def avg_feature_wasserstein(X, Y):
    # X and Y are scaled arrays, same 22 columns in same order
    # compute wasserstein distance for each feature separately, then average
    n_features = X.shape[1]
    dists = np.zeros(n_features)

    for f in range(n_features):
        dists[f] = wasserstein_distance(X[:, f], Y[:, f])

    return dists.mean()

In [13]:
domain_ids = sorted(domain_scaled.keys())
n_domains = len(domain_ids)
id_to_idx = {d: i for i, d in enumerate(domain_ids)}

wasserstein_matrix = np.zeros((n_domains, n_domains))

In [14]:
# only compute unique pairs, wasserstein per feature is symmetric so fill both sides of diag at oce
for i, j in combinations(domain_ids, 2):
    w = avg_feature_wasserstein(domain_scaled[i], domain_scaled[j])
    idx_i, idx_j = id_to_idx[i], id_to_idx[j]
    wasserstein_matrix[idx_i, idx_j] = w
    wasserstein_matrix[idx_j, idx_i] = w

print(n_domains * (n_domains - 1) // 2)

561


In [15]:
# negate distance into predicted transferability, same idea as mmd and rawdist
predicted_transfer = -wasserstein_matrix

# build dataframe
wasserstein_df = pd.DataFrame(predicted_transfer, index=domain_ids, columns=domain_ids)

In [ ]:
wasserstein_df.shape

(34, 34)

In [ ]:
wasserstein_df.head()

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,-0.000000,-0.370185,-0.821919,-0.631151,-0.271254,-0.618940,-0.601552,-0.373546,-0.467406,-0.642084,...,-0.542250,-0.748647,-0.583704,-0.311792,-0.580063,-1.038575,-0.659135,-0.584165,-0.850571,-0.426040
1,-0.370185,-0.000000,-1.059865,-0.645369,-0.345514,-0.668876,-0.784808,-0.343439,-0.598186,-0.817843,...,-0.781894,-0.947472,-0.841470,-0.268515,-0.745543,-1.256589,-0.893460,-0.560869,-1.015066,-0.557277
2,-0.821919,-1.059865,-0.000000,-1.264502,-0.902122,-1.175253,-0.597994,-1.038391,-0.615981,-0.648922,...,-0.377196,-0.731583,-0.733522,-0.924926,-0.604379,-0.606957,-0.487535,-1.092401,-1.114370,-0.855791
4,-0.631151,-0.645369,-1.264502,-0.000000,-0.666719,-0.202338,-0.915700,-0.758750,-0.769389,-1.022609,...,-1.019270,-0.670806,-0.645739,-0.676449,-0.878022,-1.320014,-1.104098,-0.402692,-0.475635,-0.629141
5,-0.271254,-0.345514,-0.902122,-0.666719,-0.000000,-0.625025,-0.503692,-0.402572,-0.462900,-0.533302,...,-0.604440,-0.826448,-0.659611,-0.315339,-0.505676,-1.002581,-0.662484,-0.643484,-0.846411,-0.512717


In [18]:
wasserstein_df.to_csv("wasserstein_transfer_matrix.csv")

In [19]:
wasserstein_T = pd.read_csv("wasserstein_transfer_matrix.csv")
wasserstein_T.set_index("Unnamed: 0", inplace=True)
wasserstein_T.index.name = None
wasserstein_T.index = wasserstein_T.index.astype(int)
wasserstein_T.columns = wasserstein_T.columns.astype(int)
wasserstein_T.head()

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,-0.000000,-0.370185,-0.821919,-0.631151,-0.271254,-0.618940,-0.601552,-0.373546,-0.467406,-0.642084,...,-0.542250,-0.748647,-0.583704,-0.311792,-0.580063,-1.038575,-0.659135,-0.584165,-0.850571,-0.426040
1,-0.370185,-0.000000,-1.059865,-0.645369,-0.345514,-0.668876,-0.784808,-0.343439,-0.598186,-0.817843,...,-0.781894,-0.947472,-0.841470,-0.268515,-0.745543,-1.256589,-0.893460,-0.560869,-1.015066,-0.557277
2,-0.821919,-1.059865,-0.000000,-1.264502,-0.902122,-1.175253,-0.597994,-1.038391,-0.615981,-0.648922,...,-0.377196,-0.731583,-0.733522,-0.924926,-0.604379,-0.606957,-0.487535,-1.092401,-1.114370,-0.855791
4,-0.631151,-0.645369,-1.264502,-0.000000,-0.666719,-0.202338,-0.915700,-0.758750,-0.769389,-1.022609,...,-1.019270,-0.670806,-0.645739,-0.676449,-0.878022,-1.320014,-1.104098,-0.402692,-0.475635,-0.629141
5,-0.271254,-0.345514,-0.902122,-0.666719,-0.000000,-0.625025,-0.503692,-0.402572,-0.462900,-0.533302,...,-0.604440,-0.826448,-0.659611,-0.315339,-0.505676,-1.002581,-0.662484,-0.643484,-0.846411,-0.512717


In [20]:
true_matrix = pd.read_csv("../transfer-matrix/rf_transfer_matrix.csv")
true_matrix.set_index("Unnamed: 0", inplace=True)
true_matrix.index.name = None
true_matrix.index = true_matrix.index.astype(int)
true_matrix.columns = true_matrix.columns.astype(int)
true_matrix.head()

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,0.392727,0.291660,0.127238,0.058840,0.214817,0.084705,0.149056,0.182358,0.173877,0.026375,...,0.228505,0.243433,0.149609,0.275203,0.112525,-0.011557,0.116862,0.158796,0.025454,0.029928
1,0.142172,0.395760,0.030643,0.010376,0.180707,0.064432,0.056092,0.128808,0.097788,0.039590,...,0.167477,0.032806,0.029704,0.193033,0.055264,0.066550,0.108203,0.199762,0.023240,-0.037710
2,0.196600,0.214445,0.368173,0.023968,0.207323,0.140139,0.172990,0.231019,0.156741,0.057226,...,0.197402,0.223793,0.121457,0.248807,0.134485,0.053503,-0.005610,0.253965,0.031899,0.059566
4,0.235489,0.227173,0.127394,0.410382,0.215954,0.186207,0.125068,0.196603,0.165264,0.034514,...,0.178055,0.245668,0.147579,0.284360,0.217076,-0.003941,0.037643,0.306750,0.137154,0.020968
5,0.190803,0.206530,0.097887,0.087230,0.464604,0.144950,0.159361,0.204884,0.264641,0.104493,...,0.197195,0.232923,0.117228,0.292623,0.170687,-0.020694,0.115293,0.297271,0.067951,0.051770


In [22]:
true_matrix.index == wasserstein_T.index

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True])

In [23]:
full_tau, _ = kendalltau(wasserstein_T.values.flatten(), true_matrix.values.flatten())
full_tau

np.float64(0.213897822225399)

In [25]:
mask = ~np.eye(len(wasserstein_T), dtype=bool)
off_diag_tau, _ = kendalltau(wasserstein_T.values[mask], true_matrix.values[mask])
print(f"off diag kendal tau: {off_diag_tau:.4f}")
# zero self distance always maps to high self transferability,
#   so it inflates overall without actually proving wasserstein ranks off diagonal pairs correctly

off diag kendal tau: 0.1665


In [ ]:
# simlar sotry to mmd, feature distribution similarity does not correlate with strong transfer performance.
# naive distribution comparison is not a good indicator of transferability